# 45 — Phase 2 Report Builder

Builds a compact markdown and JSON bundle from the 40-series outputs.
It follows the same provenance style as the earlier notebooks.

In [ ]:
import os, json, hashlib, platform, sys, math, time
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path_str):
    p = Path(path_str)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "sha256": sha256_file(p), "status": "no_rows"}
        return df, {"path": str(p), "sha256": sha256_file(p), "status": "ok"}
    except pd.errors.EmptyDataError:
        return pd.DataFrame(), {"path": str(p), "status": "empty_data"}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {
            "python": sys.version,
            "platform": platform.platform(),
        },
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

cfg = load_yaml(CONFIGS / "run.yml")
run_cfg = cfg.get("run", {})
date_from = run_cfg.get("date_from", "2025-01-01")
date_to = run_cfg.get("date_to", "2025-01-31")
sites = cfg.get("sites", [])
sites_df = pd.DataFrame(sites)
if sites_df.empty:
    sites_df = pd.DataFrame(columns=["id","name","role","lat","lon"])

In [ ]:
step = "45_phase2_report_builder"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

targets = [
    OUTPUTS / "40_target_control_extension" / "sites_registry.csv",
    OUTPUTS / "41_cross_reference_existing_outputs" / "cross_reference_daily.csv",
    OUTPUTS / "42_multi_site_comparison" / "daily_comparison.csv",
    OUTPUTS / "43_window_ranking" / "window_top_candidates.csv",
    OUTPUTS / "44_gemini_forensic_extension" / "gemini_response.json",
]

records = []
for p in targets:
    records.append({
        "path": str(p.relative_to(PROJECT_ROOT)),
        "exists": p.exists(),
        "sha256": sha256_file(p) if p.exists() else "",
        "size_bytes": p.stat().st_size if p.exists() else 0,
    })
index_df = pd.DataFrame(records)
index_df.to_csv(out / "phase2_report_index.csv", index=False)

top, _ = safe_read_csv("outputs/43_window_ranking/window_top_candidates.csv")
summary_lines = []
if not top.empty:
    for _, r in top.head(5).iterrows():
        summary_lines.append(f"- {r['window_start']} to {r['window_end']} | days={r['window_days']} | score={float(r['window_score']):.3f}")

md = []
md.append("# AQ26 Phase 2 Report Bundle")
md.append("")
md.append(f"Generated: {utcnow()}")
md.append("")
md.append("## File index")
for _, r in index_df.iterrows():
    md.append(f"- {r['path']} — {'present' if r['exists'] else 'missing'}")
md.append("")
md.append("## Top windows")
md.extend(summary_lines if summary_lines else ["- No windows available"])

(out / "phase2_report.md").write_text("\n".join(md), encoding="utf-8")

manifest = manifest_base(step, [CONFIGS / "run.yml"])
for p in [out / "phase2_report_index.csv", out / "phase2_report.md"]:
    manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})
write_json(out / "manifest.json", manifest)
display(index_df)
print("Wrote:", out)